In [1]:
import sys
print(sys.executable)

/Users/veronicachamorro/Desktop/ai_bootcamp/AI-Internship/ai-engineering-bootcamp-v2/week-1/evals/.venv/bin/python


SyntaxError: invalid syntax (1639462945.py, line 1)

In [26]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import main 
from main import retrieve_chunks,build_rag_context,build_grounding_prompt,call_model_structured
from main import DEFAULT_MODEL, EMBEDDING_MODEL, EMBEDDING_DIMENSIONS

In [6]:
# Standard library imports
import os
from pathlib import Path
from dotenv import load_dotenv

# LangChain core imports
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

# Modern LangChain agent imports (latest pattern from LangChain docs)
from langchain.agents import create_agent
from langchain.tools import tool


In [13]:
import json

GOLDEN_SET_PATH = PROJECT_ROOT / "evals" / "golden_set.json"

with open(GOLDEN_SET_PATH, "r") as f:
    golden_set = json.load(f)

print(f"Loaded {len(golden_set)} golden questions")

Loaded 5 golden questions


In [29]:
print(golden_set[0])

{'id': 'golden_001', 'question': 'How many remote days are allowed?', 'expected_answer': 'Employees may work remotely up to 3 days per week.', 'expected_document_id': 'POL-101', 'should_refuse': False}


## Chunking Retrival Eval - Manual

In [16]:
item = golden_set[0]

question = item["question"]
expected_document_id = item["expected_document_id"]

print("Question:", question)
print("Expected document:", expected_document_id)

Question: How many remote days are allowed?
Expected document: POL-101


In [18]:
retrieved_chunks = retrieve_chunks(question)

# retrieved_chunks

In [20]:
for chunk in retrieved_chunks:
    print(chunk["document_id"],"| score:",round(chunk["score"], 4),"| chunk:",chunk["id"])

POL-101 | score: 0.4455 | chunk: POL-101-6
POL-101 | score: 0.4421 | chunk: POL-101-7
POL-101 | score: 0.4287 | chunk: POL-101-4
POL-101 | score: 0.4246 | chunk: POL-101-5
POL-114 | score: 0.3766 | chunk: POL-114-4


In [21]:
retrieved_document_ids = [
    chunk["document_id"]
    for chunk in retrieved_chunks
]

retrieval_hit = expected_document_id in retrieved_document_ids

print("Expected document:", expected_document_id)
print("Retrieved documents:", retrieved_document_ids)
print("Retrieval hit:", retrieval_hit)

Expected document: POL-101
Retrieved documents: ['POL-101', 'POL-101', 'POL-101', 'POL-101', 'POL-114']
Retrieval hit: True


In [22]:
results = []

for item in golden_set:

    question = item["question"]
    expected_document_id = item["expected_document_id"]

    retrieved_chunks = retrieve_chunks(question)

    retrieved_document_ids = [
        chunk["document_id"]
        for chunk in retrieved_chunks
    ]

    if expected_document_id is None:
        retrieval_hit = None
    else:
        retrieval_hit = (
            expected_document_id in retrieved_document_ids
        )

    results.append({
        "id": item["id"],
        "question": question,
        "expected_document_id": expected_document_id,
        "retrieved_document_ids": retrieved_document_ids,
        "retrieval_hit": retrieval_hit,
    })

In [23]:
for result in results:

    print("Question:", result["question"])
    print("Expected:", result["expected_document_id"])
    print("Retrieved:", result["retrieved_document_ids"])
    print("Retrieval hit:", result["retrieval_hit"])
    print("-" * 60)

Question: How many remote days are allowed?
Expected: POL-101
Retrieved: ['POL-101', 'POL-101', 'POL-101', 'POL-101', 'POL-114']
Retrieval hit: True
------------------------------------------------------------
Question: What is the mileage rate?
Expected: POL-114
Retrieved: ['POL-114', 'POL-114', 'POL-114', 'POL-114', 'POL-101']
Retrieval hit: True
------------------------------------------------------------
Question: How quickly must a lost laptop be reported?
Expected: POL-207
Retrieved: ['POL-207', 'POL-207', 'POL-207', 'POL-207', 'POL-101']
Retrieval hit: True
------------------------------------------------------------
Question: What is the WB-9 payload limit?
Expected: SPEC-WB9
Retrieved: ['SPEC-WB9', 'SPEC-WB9', 'SPEC-WB9', 'SPEC-WB9', 'SPEC-WB9']
Retrieval hit: True
------------------------------------------------------------
Question: What is the parental leave policy?
Expected: None
Retrieved: ['POL-101', 'POL-101', 'POL-101', 'POL-101', 'POL-101']
Retrieval hit: None
-------

## LLM Answer

In [28]:
item = golden_set[0]

question = item["question"]

retrieved_chunks = retrieve_chunks(question)

context = build_rag_context(retrieved_chunks)

prompt = build_grounding_prompt(question=question,context=context,)

answer, *_ = call_model_structured(prompt,DEFAULT_MODEL,)

print(answer.answer)

Employees may work remotely up to three days per week.


In [30]:
eval_rows = []

for item in golden_set:

    question = item["question"]

    retrieved_chunks = retrieve_chunks(question)

    retrieved_document_ids = [chunk["document_id"] for chunk in retrieved_chunks]

    expected_document_id = item["expected_document_id"]

    if expected_document_id is None:
        retrieval_hit = None
    else:
        retrieval_hit = (expected_document_id in retrieved_document_ids)

    context = build_rag_context(retrieved_chunks)

    prompt = build_grounding_prompt(question=question,context=context,)

    answer, *_ = call_model_structured(prompt,DEFAULT_MODEL,)

    retrieved_contexts = [chunk["chunk_text"]for chunk in retrieved_chunks]

    eval_rows.append({
        "id": item["id"],
        "user_input": question,
        "retrieved_contexts": retrieved_contexts,
        "response": answer.answer,
        "reference": item["expected_answer"],
        "expected_document_id": expected_document_id,
        "retrieval_hit": retrieval_hit,
        "should_refuse": item["should_refuse"],
    })

In [31]:
eval_rows

[{'id': 'golden_001',
  'user_input': 'How many remote days are allowed?',
  'retrieved_contexts': ["Home office standards. Remote and hybrid employees must have a private\nspace for confidential calls, a reliable internet connection of at least\n50 Mbps down / 10 Mbps up, and an ergonomic setup that meets the Display\nScreen Equipment checklist. Northwind reimburses up to £350 (or local\nequivalent) once every 36 months for approved home office equipment.\nClaims go through the expenses policy (see POL-114).\n\nInternational remote work. Working from a country that is not the\nemployee's contracted work location for more than 10 working days in a\nrolling 90-day period requires approval from People Operations and Tax.\nUnapproved long stays can create permanent establishment and payroll\nrisks for the Company.",
   'Customer sites. Field engineers and deployment specialists follow the\ncustomer site access rules in SPEC-WB9 and the Security Policy (POL-207).\nCustomer-site days count 

## RAGAS